In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier

# Caminhos
PROJECT_ROOT = Path("..").resolve()
EXPORTS = PROJECT_ROOT / "analysis" / "exports"

df_sessions = pd.read_csv(EXPORTS / "session_metrics.csv")
df_sessions.head()


,session_id,session_start,session_end,channel_main,modes_enabled,dwell_ms,csat_mean,events_count
0,sess-0acv5ik3st,2025-10-29 22:55:36.199818+00:00,2025-10-29 22:55:37.228493+00:00,touch,"[""font_xl""]",15203.0,4.0,5
1,sess-0d5qvlh6qn,2025-10-29 22:55:32.477329+00:00,2025-10-29 22:55:35.148471+00:00,touch,[],28942.0,5.0,7
2,sess-45wzwyk169,2025-10-29 22:55:17.075396+00:00,2025-10-29 22:55:20.156145+00:00,voice,"[""high_contrast""]",27828.0,NaN,6
3,sess-4h9xilxyqm,2025-10-29 22:55:17.072938+00:00,2025-10-29 22:55:20.248636+00:00,touch,"[""high_contrast"", ""libras""]",18870.0,3.0,7
4,sess-bzw8nwyamy,2025-10-29 22:55:25.693443+00:00,2025-10-29 22:55:27.055243+00:00,touch,"[""libras"", ""font_xl""]",24294.0,3.0,5


In [2]:
df_sessions["interaction_type"] = df_sessions["dwell_ms"].apply(
    lambda x: "toque_curto" if x < 12000 else "toque_longo"
)

df_sessions["interaction_type"].value_counts()


interaction_type
toque_longo    26
toque_curto     4
Name: count, dtype: int64

In [3]:
# Converter modos para contador
df_sessions["modes_enabled"] = df_sessions["modes_enabled"].fillna("[]")

def count_modes(x):
    try:
        lst = json.loads(x)
        return len(lst) if isinstance(lst, list) else 0
    except:
        return 0

df_sessions["modes_enabled_count"] = df_sessions["modes_enabled"].apply(count_modes)

# Selecionar features finais
X = df_sessions[["channel_main", "events_count", "csat_mean", "modes_enabled_count"]]
y = df_sessions["interaction_type"]


In [4]:
encoder = OneHotEncoder(handle_unknown="ignore")
X_encoded = encoder.fit_transform(X[["channel_main"]]).toarray()

encoded_cols = encoder.get_feature_names_out(["channel_main"])

df_features = pd.concat([
    pd.DataFrame(X_encoded, columns=encoded_cols),
    X[["events_count", "csat_mean", "modes_enabled_count"]].reset_index(drop=True)
], axis=1)

df_features.head()


,channel_main_no_touch,channel_main_touch,channel_main_voice,events_count,csat_mean,modes_enabled_count
0,0.0,1.0,0.0,5,4.0,1
1,0.0,1.0,0.0,7,5.0,0
2,0.0,0.0,1.0,6,NaN,1
3,0.0,1.0,0.0,7,3.0,2
4,0.0,1.0,0.0,5,3.0,2


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.25, random_state=42
)


In [6]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [7]:
acc = accuracy_score(y_test, y_pred)
acc

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

 toque_curto       0.00      0.00      0.00         1
 toque_longo       0.83      0.71      0.77         7

    accuracy                           0.62         8
   macro avg       0.42      0.36      0.38         8
weighted avg       0.73      0.62      0.67         8



In [8]:
from sklearn.metrics import accuracy_score, f1_score

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", acc)
print("F1-score:", f1)

Accuracy: 0.625
F1-score: 0.6730769230769231


In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
cm_df = pd.DataFrame(cm, index=[f"real_{c}" for c in model.classes_], columns=[f"pred_{c}" for c in model.classes_])
cm_df

,pred_toque_curto,pred_toque_longo
real_toque_curto,0,1
real_toque_longo,2,5


In [10]:
feature_importances = pd.DataFrame({
    "feature": df_features.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importances


,feature,importance
3,events_count,0.523810
4,csat_mean,0.305952
1,channel_main_touch,0.104762
2,channel_main_voice,0.065476
0,channel_main_no_touch,0.000000
5,modes_enabled_count,0.000000
